# 04 — Signal Quality

Evaluates the MA-200 signal as a series of **independent bets**: each entry-to-exit
cycle is treated like a single trade, so signal quality is measured by how often the
signal is right and how large wins are relative to losses — not by cumulative NAV.

**Pipeline**
```
YahooFinanceProvider  →  TrendSignal  →  BarBacktest  →  SignalAnalytics  →  SignalComparison
      (bars)            (signal cols)   (bt_result)       (per-trade)        (variant A vs B)
```

In [1]:
import pandas as pd
import plotly.graph_objects as go

from hailmary.data.providers import YahooFinanceProvider
from hailmary.models import TrendSignal
from hailmary.backtest.signal_backtest import BarBacktest
from hailmary.analytics.signal_analytics import SignalAnalytics, TradeQuality
from hailmary.analytics.signal_comparison import SignalComparison
from hailmary.viz.theme import PALETTE, apply_theme

## 1. Data → Signal → Backtest

In [2]:
signal = TrendSignal(ma_window=200)
yahoo  = YahooFinanceProvider()
symbols = ["BTC-USD", "ETH-USD", "SOL-USD"]
start, end = pd.Timestamp("2022-01-01"), pd.Timestamp("2024-01-01")

fetch_start = start - pd.offsets.BDay(signal.warmup)
bars = yahoo.get_bars(symbols, start=fetch_start, end=end, adjust=False)

signal_df = signal.run(bars, trim_start=start)
bt_result = BarBacktest().run(signal_df)
analytics = SignalAnalytics(bt_result)

print(f"{bt_result.data.shape[0]:,} bars  |  {bt_result.data.index.get_level_values('symbol').nunique()} symbols")

2026-05-01 19:56:03.273 | DEBUG    | hailmary.data.cache:get:38 - Cache hit key=de1ec4900549


2,193 bars  |  3 symbols


## 2. Per-Trade Quality Table

`SignalAnalytics.trade_summary()` treats every entry-to-exit cycle as an **independent bet**
and aggregates trade-level outcomes — the right unit for evaluating whether the signal has
edge, independent of how long you happened to be in the market.

| Group | Metrics | What they answer |
|---|---|---|
| **Win / Loss** | `win_rate`, `avg_win`, `avg_loss` | Do winners outsize losers, and how often? |
| **Edge** | `expectancy`, `exp_ex_top`, `median`, `profit_factor` | Is there positive expected value per bet, and is it robust to outliers? |
| **Tail Shape** | `max_win`, `max_loss`, `skewness` | How extreme are the tails; is the distribution symmetric? |
| **Duration** | `avg_duration` | How long does the signal keep you deployed? |
| **Intra-Trade DD** | `avg_intra_drawdown`, `max_intra_drawdown` | How much peak-to-trough pain within a cycle? |

Both **Net** (after round-trip cost) and **Conservative** (worst-case fill) are shown side by side.
The **Flags** column auto-detects outlier-dependence and distributional warnings.

In [3]:
from IPython.display import HTML as _HTML

ts_net = analytics.trade_summary(method="net")
ts_con = analytics.trade_summary(method="conservative")


def _bg(val: float, bound: float) -> str:
    if pd.isna(val) or bound == 0:
        return ""
    t = max(-1.0, min(1.0, float(val) / bound))
    if t >= 0:
        r = b = round(255 * (1 - 0.75 * t))
        g = round(255 * (1 - 0.15 * t))
    else:
        t = -t
        r = round(255 * (1 - 0.15 * t))
        g = b = round(255 * (1 - 0.75 * t))
    return f"background-color: rgb({r},{g},{b}); color: #111;"


def _bg_red(val: float, bound: float) -> str:
    if pd.isna(val) or bound == 0:
        return ""
    t = min(1.0, abs(float(val) / bound))
    r = round(255 * (1 - 0.15 * t))
    g = b = round(255 * (1 - 0.75 * t))
    return f"background-color: rgb({r},{g},{b}); color: #111;"


_NOTE_CSS = (
    "font-size:11px; color:#999; margin-top:5px; "
    "padding:4px 8px; line-height:1.8; border-top:1px solid #444;"
)


def _with_footnotes(styler, notes: list[str]) -> _HTML:
    lines = "<br>".join(notes)
    footer = f'<div style="{_NOTE_CSS}">{lines}</div>'
    return _HTML(styler.to_html() + footer)


def _outlier_flag(row: pd.Series) -> str:
    flags = TradeQuality.quality_flags(
        expectancy=row[("Net", "Expectancy")],
        expectancy_ex_top=row[("Net", "Exp ex-Top")],
        median_return=row[("Net", "Median")],
        skewness=row[("Net", "Skewness")],
    )
    parts = []
    if flags["median_negative"]:   parts.append("⚠ median < 0")
    if flags["top_trade_outlier"]: parts.append("⚠ top trade >30% of edge")
    if flags["skewed_right"]:      parts.append("↑ skewed right")
    elif flags["skewed_left"]:     parts.append("↓ skewed left")
    return ", ".join(parts) if parts else "—"


_METRICS = ["win_rate", "avg_win", "avg_loss", "expectancy", "expectancy_ex_top",
            "median_return", "profit_factor", "skewness", "avg_intra_drawdown", "max_intra_drawdown"]

display_df = pd.concat([
    ts_net[["n_trades"]],
    ts_net[_METRICS],
    ts_con[_METRICS].rename(columns=lambda c: f"{c}_con"),
    ts_net[["avg_duration"]],
], axis=1)

_cols = ["Win Rate", "Avg Win", "Avg Loss", "Expectancy", "Exp ex-Top",
         "Median", "Profit Factor", "Skewness", "Avg DD", "Worst DD"]
display_df.columns = pd.MultiIndex.from_tuples(
    [("Bets", "Count")]
    + [("Net", c) for c in _cols]
    + [("Conservative", c) for c in _cols]
    + [("Duration", "Avg (bars)")]
)
display_df[("", "Flags")] = [_outlier_flag(display_df.loc[sym]) for sym in display_df.index]

_pct  = "{:+.1%}"
_pct0 = "{:.1%}"
_f2   = "{:.2f}"
fmt = {
    ("Bets",         "Count"):       "{:.0f}",
    ("Duration",     "Avg (bars)"): "{:.0f}",
    **{("Net",          c): _pct  for c in ["Avg Win", "Avg Loss", "Expectancy", "Exp ex-Top", "Median", "Avg DD", "Worst DD"]},
    **{("Conservative", c): _pct  for c in ["Avg Win", "Avg Loss", "Expectancy", "Exp ex-Top", "Median", "Avg DD", "Worst DD"]},
    ("Net",          "Win Rate"):    _pct0,
    ("Conservative", "Win Rate"):    _pct0,
    ("Net",          "Profit Factor"): _f2,
    ("Conservative", "Profit Factor"): _f2,
    ("Net",          "Skewness"):    "{:+.2f}",
    ("Conservative", "Skewness"):    "{:+.2f}",
}

styler = display_df.style.format(fmt).set_caption(
    "MA-200 Trend Signal — Per-Trade Quality (Net vs Conservative)"
)

exp_bound = max(
    abs(float(display_df[[("Net", "Expectancy"), ("Conservative", "Expectancy")]].min().min())),
    abs(float(display_df[[("Net", "Expectancy"), ("Conservative", "Expectancy")]].max().max())),
    0.001,
)
for col in [("Net", "Expectancy"), ("Net", "Exp ex-Top"), ("Net", "Median"),
            ("Conservative", "Expectancy"), ("Conservative", "Exp ex-Top"), ("Conservative", "Median")]:
    styler = styler.map(lambda v, b=exp_bound: _bg(v, b), subset=[col])

for col in [("Net", "Win Rate"), ("Conservative", "Win Rate")]:
    styler = styler.map(lambda v: _bg(float(v) - 0.5, 0.5) if not pd.isna(v) else "", subset=[col])

for col in [("Net", "Profit Factor"), ("Conservative", "Profit Factor")]:
    styler = styler.map(
        lambda v: _bg(min(float(v), 3.0) - 1.0, 2.0)
        if not pd.isna(v) and v != float("inf") else "",
        subset=[col],
    )

dd_bound = max(
    abs(float(display_df[[("Net", "Worst DD"), ("Conservative", "Worst DD")]].min().min())),
    0.001,
)
for col in [("Net", "Avg DD"), ("Net", "Worst DD"),
            ("Conservative", "Avg DD"), ("Conservative", "Worst DD")]:
    styler = styler.map(lambda v, b=dd_bound: _bg_red(v, b), subset=[col])

_border = "border-left: 2px solid #888 !important;"
styler = styler.set_table_styles(
    [{"selector": sel, "props": _border} for sel in [
        "th.col_heading.level0.col1",  "th.col_heading.level1.col1",  "td.col1",
        "th.col_heading.level0.col11", "th.col_heading.level1.col11", "td.col11",
        "th.col_heading.level0.col21", "th.col_heading.level1.col21", "td.col21",
        "th.col_heading.level0.col22", "th.col_heading.level1.col22", "td.col22",
    ]],
    overwrite=False,
)

_with_footnotes(styler, [
    "<b>Win Rate</b> — fraction of completed trades that closed with a positive compound return.",
    "<b>Expectancy</b> — win_rate × avg_win + (1 − win_rate) × avg_loss; expected return per bet. Positive = edge exists.",
    "<b>Exp ex-Top</b> — expectancy after removing the single best trade. Large gap vs Expectancy = one outlier drives most of the edge.",
    "<b>Median</b> — 50th-percentile trade return. More robust than the mean; Median ≪ Expectancy = right-skewed distribution.",
    "<b>Profit Factor</b> — Σ winning returns / |Σ losing returns|; > 1 means the strategy earns more than it loses. ∞ = no losing trades.",
    "<b>Skewness</b> — > +1: long right tail (rare large wins); < −1: long left tail (rare large losses).",
    "<b>Avg DD / Worst DD</b> — intra-trade peak-to-trough, entry price as initial peak. Always ≤ 0.",
    "<b>Net</b> = MTC fill (open after signal bar) minus round-trip cost. "
    "<b>Conservative</b> = worst-case fill (high entry / low exit) — no cost deducted. "
    "Both use the same colour scale so values are directly comparable.",
    "<b>Flags</b> — ⚠ median &lt; 0 = mean lifted by outliers; ⚠ top trade &gt;30% of edge = fragile; ↑/↓ skewed = asymmetric distribution.",
])

## 3. Aligned Trade Paths

Every trade pivoted to **day 0 = entry** so all paths start at 0%.  X-axis is bars
since entry; Y-axis is cumulative return from the entry price.

Green = trade ended in profit for that fill method, red = loss.  Hover any line to
see ticker, dates, duration, and both final returns.  The **bold yellow** line is
the mean, **dashed purple** is the median, and the shaded band is ±1σ — all
computed only from trades still active at each bar.

In [4]:
import numpy as np

trade_paths = TradeQuality(analytics).trade_paths()
print(f"{len(trade_paths)} trades  |  {len(set(p['sym'] for p in trade_paths))} symbols")

17 trades  |  3 symbols


In [5]:
def _aligned_chart(paths: list, cum_key: str, title: str,
                   avg_colour: str, band_rgba: str) -> go.Figure:
    fig = go.Figure()
    shown: set[str] = set()

    for t in paths:
        cum    = t[cum_key]
        win    = float(cum.iloc[-1]) > 0
        colour = PALETTE["accent_green"] if win else PALETTE["accent_red"]
        label  = "Win" if win else "Loss"
        show   = label not in shown
        if show:
            shown.add(label)
        n      = len(cum)
        custom = [[
            t["label"],
            str(t["entry_dt"].date()),
            str(t["exit_dt"].date()),
            t["duration"],
            f"{t['final_net']:+.1%}",
            f"{t['final_con']:+.1%}",
            f"{t['max_dd_net']:.1%}",
            f"{t['max_dd_con']:.1%}",
        ]] * n
        fig.add_trace(go.Scatter(
            x=list(range(n)),
            y=(cum * 100).tolist(),
            mode="lines",
            line=dict(color=colour, width=1.5),
            opacity=0.55,
            name=label, legendgroup=label, showlegend=show,
            customdata=custom,
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "Entry: %{customdata[1]}  →  Exit: %{customdata[2]}<br>"
                "Duration: %{customdata[3]} bars<br>"
                "Day %{x}: <b>%{y:.1f}%</b><br>"
                "Final — net: %{customdata[4]}  |  conservative: %{customdata[5]}<br>"
                "Max DD — net: %{customdata[6]}  |  conservative: %{customdata[7]}"
                "<extra></extra>"
            ),
        ))

    max_dur = max(len(t[cum_key]) for t in paths)
    avgs, meds, stds = [], [], []
    for d in range(max_dur):
        vals = [float(t[cum_key].iloc[d]) * 100 for t in paths if d < len(t[cum_key])]
        avgs.append(np.mean(vals))
        meds.append(np.median(vals))
        stds.append(np.std(vals))

    days  = list(range(max_dur))
    upper = [m + s for m, s in zip(avgs, stds)]
    lower = [m - s for m, s in zip(avgs, stds)]

    fig.add_trace(go.Scatter(
        x=days + days[::-1], y=upper + lower[::-1],
        fill="toself", fillcolor=band_rgba,
        line=dict(color="rgba(0,0,0,0)"),
        name="±1σ", hoverinfo="skip",
    ))
    fig.add_trace(go.Scatter(
        x=days, y=avgs,
        mode="lines", line=dict(color=avg_colour, width=2.5),
        name="Mean",
        hovertemplate="Day %{x}<br>Mean: <b>%{y:.1f}%</b><extra></extra>",
    ))
    fig.add_trace(go.Scatter(
        x=days, y=meds,
        mode="lines", line=dict(color=PALETTE["accent_purple"], width=2, dash="dash"),
        name="Median",
        hovertemplate="Day %{x}<br>Median: <b>%{y:.1f}%</b><extra></extra>",
    ))

    fig.add_hline(y=0, line=dict(color=PALETTE["text_secondary"], width=0.8, dash="dash"))
    apply_theme(fig, title=title, height=480)
    fig.update_layout(xaxis_title="Bars since entry", yaxis_title="Cumulative return from entry (%)")
    return fig


_aligned_chart(
    trade_paths, "cum_net",
    title="Aligned Trade Paths — Net Fill",
    avg_colour=PALETTE["accent_yellow"],
    band_rgba="rgba(255,215,0,0.10)",
).show()

In [6]:
_aligned_chart(
    trade_paths, "cum_con",
    title="Aligned Trade Paths — Conservative Fill",
    avg_colour=PALETTE["accent_orange"],
    band_rgba="rgba(255,140,0,0.10)",
).show()

In [7]:
tbl = pd.DataFrame([{
    "Trade":        t["label"],
    "Entry":        t["entry_dt"].date(),
    "Exit":         t["exit_dt"].date(),
    "Days":         t["duration"],
    "Net":          t["final_net"],
    "Conservative": t["final_con"],
    "Max DD (Net)": t["max_dd_net"],
    "Max DD (Con)": t["max_dd_con"],
    "_win":         t["final_net"] > 0,
} for t in trade_paths])

_fmt = {
    "Net":          "{:+.1%}",
    "Conservative": "{:+.1%}",
    "Max DD (Net)": "{:.1%}",
    "Max DD (Con)": "{:.1%}",
}

def _style_tbl(df: pd.DataFrame, header_colour: str) -> object:
    ret_bound = max(abs(df["Net"].max()), abs(df["Net"].min()), 0.001)
    def _ret_bg(v: float) -> str:
        t = max(-1.0, min(1.0, v / ret_bound))
        if t >= 0:
            r = b = round(255 * (1 - 0.75 * t)); g = round(255 * (1 - 0.15 * t))
        else:
            t = -t; r = round(255 * (1 - 0.15 * t)); g = b = round(255 * (1 - 0.75 * t))
        return f"background-color: rgb({r},{g},{b}); color: #111;"
    def _dd_bg(v: float) -> str:
        bound = max(abs(df[["Max DD (Net)", "Max DD (Con)"]].min().min()), 0.001)
        t = min(1.0, abs(v) / bound)
        r = round(255 * (1 - 0.15 * t)); g = b = round(255 * (1 - 0.75 * t))
        return f"background-color: rgb({r},{g},{b}); color: #111;"
    return (
        df.style.format(_fmt)
        .map(_ret_bg, subset=["Net", "Conservative"])
        .map(_dd_bg,  subset=["Max DD (Net)", "Max DD (Con)"])
        .set_table_styles([{"selector": "thead tr th",
                            "props": f"background-color: {header_colour}; color: #111; font-weight: bold;"}])
    )

wins   = tbl[ tbl["_win"]].drop(columns="_win").reset_index(drop=True)
losses = tbl[~tbl["_win"]].drop(columns="_win").reset_index(drop=True)

print(f"Wins ({len(wins)})")
display(_style_tbl(wins, "#2ea043"))
print(f"\nLosses ({len(losses)})")
display(_style_tbl(losses, "#da3633"))

Wins (5)


,Trade,Entry,Exit,Days,Net,Conservative,Max DD (Net),Max DD (Con)
0,BTC-USD T1,2023-01-14,2023-08-18,217,+33.8%,+21.8%,-18.7%,-18.7%
1,BTC-USD T3,2023-10-17,2024-01-01,77,+54.9%,+54.3%,-6.6%,-6.6%
2,ETH-USD T3,2023-01-13,2023-08-18,218,+18.6%,+12.5%,-22.1%,-22.4%
3,ETH-USD T5,2023-10-30,2024-01-01,64,+31.0%,+28.6%,-8.7%,-8.7%
4,SOL-USD T9,2023-10-01,2024-01-01,93,+411.9%,+348.9%,-20.7%,-20.7%



Losses (12)


,Trade,Entry,Exit,Days,Net,Conservative,Max DD (Net),Max DD (Con)
0,BTC-USD T2,2023-08-30,2023-08-31,2,-1.5%,-7.2%,-1.5%,-7.2%
1,ETH-USD T1,2022-01-01,2022-01-08,8,-13.3%,-19.9%,-16.6%,-21.1%
2,ETH-USD T2,2022-04-04,2022-04-06,3,-3.1%,-10.3%,-3.1%,-10.3%
3,ETH-USD T4,2023-10-26,2023-10-28,3,-0.4%,-4.9%,-1.3%,-4.9%
4,SOL-USD T1,2022-01-01,2022-01-20,20,-20.3%,-28.9%,-23.9%,-28.9%
5,SOL-USD T2,2023-02-21,2023-02-22,2,-4.7%,-11.5%,-4.7%,-11.5%
6,SOL-USD T3,2023-04-12,2023-05-25,44,-16.5%,-22.4%,-24.2%,-25.5%
7,SOL-USD T4,2023-05-26,2023-06-08,14,-3.2%,-6.4%,-14.6%,-16.2%
8,SOL-USD T5,2023-07-08,2023-08-19,43,-0.7%,-4.9%,-22.1%,-22.2%
9,SOL-USD T6,2023-08-20,2023-08-22,3,-3.1%,-10.3%,-3.1%,-10.3%


## 4. Return Distribution

Histogram of net trade returns per symbol — green bars are winning trades, red are losing.

Vertical lines show where the key summary stats land against the full distribution:
- **Yellow solid** — net expectancy (mean)
- **Orange dotted** — conservative expectancy
- **Purple dashed** — net median

If yellow is far to the right of purple, a few large winners are inflating the mean — consistent with the Exp ex-Top flag in the table above.

In [8]:
from plotly.subplots import make_subplots

trades  = analytics.trade_stats()
symbols = sorted(trades["symbol"].unique())
n_sym   = len(symbols)

fig = make_subplots(
    rows=1, cols=n_sym,
    subplot_titles=symbols,
    shared_yaxes=True,
    horizontal_spacing=0.06,
)

# Invisible legend entries for the vlines
for colour, dash, label in [
    (PALETTE["accent_yellow"], "solid", "Expectancy — Net"),
    (PALETTE["accent_orange"], "dot",   "Expectancy — Conservative"),
    (PALETTE["accent_purple"], "dash",  "Median — Net"),
]:
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode="lines",
        line=dict(color=colour, width=2, dash=dash),
        name=label,
    ))

shown: set[str] = set()
for col_i, sym in enumerate(symbols, start=1):
    sym_trades = trades[trades["symbol"] == sym]
    ret_net    = sym_trades["return_net"]

    for data, colour, label in [
        (ret_net[ret_net <= 0], PALETTE["accent_red"],   "Loss"),
        (ret_net[ret_net >  0], PALETTE["accent_green"], "Win"),
    ]:
        show = label not in shown
        if show:
            shown.add(label)
        fig.add_trace(go.Histogram(
            x=data,
            name=label, legendgroup=label, showlegend=show,
            marker_color=colour, opacity=0.8,
            nbinsx=8,
        ), row=1, col=col_i)

    exp_net = float(ts_net.loc[sym, "expectancy"])
    exp_con = float(ts_con.loc[sym, "expectancy"])
    med_net = float(ts_net.loc[sym, "median_return"])

    fig.add_vline(x=0,       line_dash="dash",  line_color=PALETTE["text_secondary"], line_width=1,   row=1, col=col_i)
    fig.add_vline(x=exp_net, line_dash="solid", line_color=PALETTE["accent_yellow"],  line_width=2,   row=1, col=col_i)
    fig.add_vline(x=exp_con, line_dash="dot",   line_color=PALETTE["accent_orange"],  line_width=2,   row=1, col=col_i)
    fig.add_vline(x=med_net, line_dash="dash",  line_color=PALETTE["accent_purple"],  line_width=2,   row=1, col=col_i)

fig.update_layout(barmode="overlay")

for i in range(1, n_sym + 1):
    axis = f"xaxis{'' if i == 1 else i}"
    fig.update_layout(**{axis: dict(tickformat=".0%", title_text="Return per trade")})
fig.update_layout(yaxis_title="# Trades")

apply_theme(fig, title="Trade Return Distribution — Net Fill", height=420)
fig.show()

## 5. Signal Comparison Harness

Add variants to the `variants` dict as you iterate — different MA windows, regime
filters, mixed signals, etc.  Re-run this section to see all variants side by side.

The tearsheet has three panels:

1. **Pooled quality table** — trades pooled across all symbols.  Net columns: WR,
   Avg Win/Loss, Expectancy, **Exp ex-Top** (edge without the best trade),
   **Median**, Profit Factor, **Skewness**, Avg/Worst intra-trade DD.  Conservative
   columns mirror the key edge metrics for fill-risk comparison.  Use Exp ex-Top and
   Median to spot variants whose edge is driven by a single outlier trade.
2. **Per-symbol expectancy heatmap** — net expectancy per (variant × symbol).  Spot
   where a change helps one asset but hurts another, so you can decide whether to
   drop the asset or tune per-asset.
3. **Return distribution** — box + individual-trade scatter per symbol, one box per
   variant.  Shows whether a variant shifts the distribution right (better edge) or
   tightens it (more consistent).

In [9]:
# Add new variants here as you iterate.  Each value is a BarBacktestResult.
# Example:
#   bt_ma100  = BarBacktest().run(TrendSignal(ma_window=100).run(bars))
#   bt_ma50   = BarBacktest().run(TrendSignal(ma_window=50).run(bars))
variants = {
    "MA-200 Base": bt_result,
    # "MA-100":    bt_ma100,
    # "MA-50":     bt_ma50,
}

SignalComparison(variants).tearsheet().show()

## 6. Standalone HTML Tearsheet

`SignalTearsheet` bundles everything above — portfolio equity + drawdown, summary
metrics, per-trade quality table, aligned trade paths, and return distribution —
into a single self-contained `.html` file that opens in any browser without Jupyter.

```python
SignalTearsheet(analytics, title="MA-200 — BTC/ETH/SOL 2022-2024").save("reports/ma200.html")
```

In [10]:
from hailmary.viz.signal_tearsheet import SignalTearsheet
from pathlib import Path

path = SignalTearsheet(
    analytics,
    title="MA-200 Trend Signal — BTC/ETH/SOL 2022–2024",
    mode="rebalanced",          # "rebalanced" | "buy_and_hold" | "fixed_stake"
    # method defaults to ["net", "conservative"]
    # amount_per_entry=1_000,   # only used when mode="fixed_stake"
).save(Path("../../reports/ma200_tearsheet.html"), open=True)

print(f"Saved → {path}")

Saved → C:\Users\Dalva\src\project-hail-mary\reports\ma200_tearsheet.html
